# 🙏 BrajYatra AI — Kaggle GPU Fallback Server

This notebook runs **Gemma 3 (4B)** on Kaggle's free T4 GPU and exposes it as a REST API via ngrok.

## Setup
1. Enable **GPU T4 x2** in Kaggle notebook settings
2. Add your ngrok authtoken as a Kaggle Secret named `NGROK_TOKEN`
3. Run all cells
4. Copy the ngrok URL and set it as `KAGGLE_ENDPOINT_URL` in your `.env` file
5. Set `LLM_MODE=hybrid` in `.env` to enable fallback

In [ ]:
# Install dependencies
!pip install -q flask pyngrok transformers accelerate torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-4b-it"

print(f"Loading {MODEL_ID}...")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Model loaded!")

In [ ]:
def generate_text(system_prompt, message, history=None, max_tokens=2048):
    """Generate response using Gemma."""
    messages = []
    
    # Add system context as first user message  
    if system_prompt:
        messages.append({"role": "user", "content": f"System instruction: {system_prompt}"})
        messages.append({"role": "assistant", "content": "Understood. I will follow these instructions."})
    
    # Add history
    if history:
        for msg in history:
            role = msg.get('role', 'user')
            if role == 'assistant': role = 'assistant'
            messages.append({"role": role, "content": msg.get('content', '')})
    
    # Add current message
    messages.append({"role": "user", "content": message})
    
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

# Quick test
test = generate_text("", "Hello! Say 'BrajYatra ready!' if you can hear me.", max_tokens=50)
print(f"Test: {test}")

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'gemma-3-4b-it', 'gpu': str(torch.cuda.get_device_name(0))})

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.json
        system_prompt = data.get('system_prompt', '')
        message = data.get('message', '')
        history = data.get('history', [])
        
        if not message:
            return jsonify({'error': 'message is required'}), 400
        
        response = generate_text(system_prompt, message, history)
        return jsonify({'text': response, 'model': 'gemma-3-4b-it'})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

print("Flask app ready.")

In [ ]:
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# Get ngrok token from Kaggle secrets
try:
    secrets = UserSecretsClient()
    ngrok_token = secrets.get_secret("NGROK_TOKEN")
    ngrok.set_auth_token(ngrok_token)
except:
    print("⚠️ NGROK_TOKEN not found in Kaggle Secrets.")
    print("Add it via: Add-ons > Secrets > NGROK_TOKEN")
    print("Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken")

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print("="*60)
print(f"🌐 PUBLIC URL: {public_url}")
print(f"")
print(f"Set this in your .env file:")
print(f"KAGGLE_ENDPOINT_URL={public_url}")
print(f"LLM_MODE=hybrid")
print("="*60)

# Run Flask (blocking)
app.run(host='0.0.0.0', port=5000)